# Trial-order shuffle control: full shuffled fit

**Objective.** Audit sequence boundaries, create one deterministic row-wise shuffle within missing-delimited segments, and fit the final revised 3-state soft EM-HMM to all shuffled subjects.

## Contract

`p_*` are smoothed forward-backward state posteriors. `filtered_prob_*` use data through trial t. `prior_predictive_prob_*` and `one_step_predictive_*` are response-before predictions. Training likelihood is descriptive, not evidence of generalization.

In [1]:
from pathlib import Path
import sys, json, pandas as pd
WORKSPACE=Path.cwd()
RUNTIME=WORKSPACE/'tmp'/'jupyter-notebook'/'runtime'
CORE=WORKSPACE/'shuffle_control'/'model_artifacts'
for p in (RUNTIME,CORE):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
from shuffle_control_core import protected_snapshot, prepare_data, run_full_fit
before=protected_snapshot()
print({'workspace':str(WORKSPACE),'runtime':RUNTIME.exists()})

{'workspace': 'D:\\neuromatch_pod_work\\HMM_me', 'runtime': True}


## Block audit and deterministic shuffled dataset

In [2]:
prepared=prepare_data(force=True)
display(pd.Series(prepared['audit'],name='value').to_frame())
display(prepared['block_report'].head())
display(prepared['missing'])

,value
block_definition,subject_id + session_id + run_id
shuffle_definition,subject_id + original_block_id + missing-delim...
number_of_subjects,12
number_of_subject_session_pairs,81
number_of_global_run_id_values,44
number_of_model_blocks,389
number_of_valid_segments,388
block_length_min,1
block_length_median,215.0
block_length_max,226


,block_definition,subject_id,experiment_id,session_id,run_id,original_block_id,segment_id,shuffled_block_id,stable_block_number,shuffle_seed,n_trials_before,n_trials_after,order_changed,prior_std_unique_count,prior_mean_unique_count
0,subject_id + session_id + run_id; then split a...,1,11,1,1,s01_session001_run001,s01_session001_run001_segment01,s01_session001_run001_segment01,0,20260717,213,213,True,1,1
1,subject_id + session_id + run_id; then split a...,1,11,1,2,s01_session001_run002,s01_session001_run002_segment01,s01_session001_run002_segment01,1,20260718,215,215,True,1,1
2,subject_id + session_id + run_id; then split a...,1,11,1,3,s01_session001_run003,s01_session001_run003_segment01,s01_session001_run003_segment01,2,20260719,215,215,True,1,1
3,subject_id + session_id + run_id; then split a...,1,11,1,4,s01_session001_run004,s01_session001_run004_segment01,s01_session001_run004_segment01,3,20260720,207,207,True,1,1
4,subject_id + session_id + run_id; then split a...,1,11,1,5,s01_session001_run005,s01_session001_run005_segment01,s01_session001_run005_segment01,4,20260721,215,215,True,1,1


,subject_id,experiment_id,experiment_name,session_id,run_id,trial_index,prior_std,prior_mean,motion_coherence,motion_direction,estimate_x,estimate_y,original_row_index,original_block_id,missing_reason
30043,4,11,data01_direction4priors,5,20,146,80,225,0.06,305,NaN,NaN,30043,s04_session005_run020,estimate_x_or_estimate_y_missing
30651,4,11,data01_direction4priors,5,23,188,80,225,0.06,275,NaN,NaN,30651,s04_session005_run023,estimate_x_or_estimate_y_missing
66604,10,12,data01_direction4priors,3,12,1,20,225,0.06,225,NaN,NaN,66604,s10_session003_run012,estimate_x_or_estimate_y_missing


The model block is `subject_id + session_id + run_id`; all three conditioning fields are fixed within it. Missing-response trials are boundaries. The actual permutation unit is the resulting segment, using master seed 20260717 plus a stable sorted segment number.

## Fit all shuffled subjects with the unchanged revised Soft EM-HMM

In [3]:
result=run_full_fit(force=True)
display(result['summary'])

full subject 01: iter=213 converged=True


full subject 02: iter=300 converged=False


full subject 03: iter=130 converged=True


full subject 04: iter=130 converged=True


full subject 05: iter=267 converged=True


...

full subject 09: iter=300 converged=False


full subject 10: iter=300 converged=False


full subject 11: iter=106 converged=True


full subject 12: iter=87 converged=True


,subject_id,n_trials,n_original_blocks,n_segments,converged,n_iterations,initial_log_likelihood,final_log_likelihood,log_likelihood_improvement,likelihood_monotonic,...,kappa_s_24,kappa_p_10,kappa_p_20,kappa_p_40,kappa_p_80,state_collapse_flag,has_nan,has_inf,reached_max_iter,runtime_seconds
0,1,8562,40,40,True,213,-5639.812393,-4015.085564,1624.726829,True,...,30.134137,133.955614,43.797499,15.530497,1.160210,False,False,False,False,11.243431
1,2,7877,37,37,False,300,-6460.597054,-4641.868006,1818.729048,True,...,17.142732,74.503518,32.159391,7.488730,1.971813,False,False,False,True,14.417955
2,3,9412,44,44,True,130,-5993.713158,-4485.464809,1508.248349,True,...,26.597790,26.897044,14.005183,11.067061,55.873345,False,False,False,False,7.677733
3,4,4799,23,23,True,130,-4239.370759,-3798.470751,440.900008,True,...,10.880614,17.576537,11.087157,4.010438,2.182009,False,False,False,False,3.938562
4,5,5789,27,27,True,267,-8075.578559,-7412.483467,663.095092,True,...,9.299115,358.137559,0.000001,1.917610,12.711175,False,False,False,False,9.542925
5,6,7553,35,35,True,201,-7537.697058,-6997.212588,540.484471,True,...,13.323541,10.321067,7.602414,3.566519,0.000001,False,False,False,False,9.387184
6,7,5797,27,27,True,142,-4376.773947,-3832.054357,544.719590,True,...,16.035162,23.864199,10.152181,4.238857,1.058772,False,False,False,False,5.166973
7,8,5797,27,27,True,249,-7708.016001,-7173.452503,534.563497,True,...,8.915597,7.889878,4.068916,3.313331,2.065452,False,False,False,False,8.899959
8,9,8632,40,40,False,300,-8054.107889,-7478.426144,575.681745,True,...,18.794216,19.321377,15.714808,1.224927,0.976290,False,False,False,True,15.823075
9,10,6044,28,28,False,300,-6722.465277,-6362.610490,359.854787,True,...,9.180760,30.913335,25.237017,9.081451,0.323409,False,False,False,True,11.156320


## Full-fit validation

In [4]:
validation=json.loads((WORKSPACE/'shuffle_control'/'logs'/'full_fit_validation.json').read_text())
display(pd.Series(validation,name='passed').to_frame())
assert all(validation.values())

,passed
row_count_preserved,True
original_trial_once,True
within_segment_only,True
most_nontrivial_segments_changed,True
gamma_sums_one,True
finite_model_outputs,True
transition_rows_sum_one,True
all_subjects_succeeded,True


## Interpretation boundary

The full-data likelihood and smoothed posterior summaries describe the fit. Predictive conclusions are deferred to the separate held-out notebook.